[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/08_developability/08_dev_lab.ipynb)


# 08 — developability (liability scan 직접 실행)

> 본문: [`08_developability.md`](08_developability.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 3초** (실측)
> **앞 랩에서 이어져요** — Ch.04 numbering 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.

**이 노트북은 도구를 직접 돌립니다.** `liability_scan.py` 를 **직접 돌려** 스캔 결과를 `my_run/` 에 만들고 커밋본과 대조해요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "08_developability"
PIP_PKGS = "pandas matplotlib biopython"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = False    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")           # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨집니다

if _APT:
    _run("apt-get -qq update", quiet=True)   # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), quiet=True)

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — liability scan (본문 8.1)

```bash
python scripts/liability_scan.py data/demo_mab.fa --out my_run/liability.csv
```
motif 정규식 4종(N-glyc sequon `N-X-S/T` · deamidation `NG`·`NS` · isomerization `DG`)과
pI·GRAVY·Cys 홀짝·Met/Trp 개수를 한 번에 계산해요.
motif 컬럼에는 **hit 위치(1-based, 세미콜론 구분)** 가 들어가고, hit 이 없으면 빈 칸입니다.

In [ ]:
run_tool([PY, SCRIPTS/"liability_scan.py", "data/demo_mab.fa",
          "--out", "my_run/liability.csv"])

## 2) 내 결과 확인 — 물리화학·Cys·산화 지표 (본문 8.2)

**값이 하나라도 채워진 컬럼만** 표에 세웁니다. 두 사슬 모두 빈 컬럼을 그대로 보여주면
"측정했는데 0" 인지 "아예 비어 있는지"를 구분할 수 없거든요.

In [ ]:
import pandas as pd
df = pd.read_csv(resolve("liability.csv"))

PHYS  = ["id", "length", "molecular_weight", "pI", "gravy", "cysteine_count",
         "odd_cysteine_flag", "methionine_count", "tryptophan_count", "ambiguous_residues"]
MOTIF = ["N_glycosylation_NXS_T", "deamidation_NG", "deamidation_NS", "isomerization_DG"]

def filled(frame, wanted):
    """있는 컬럼 중 값이 하나라도 채워진 것만 고른다(전부 빈 컬럼은 표에서 뺀다)."""
    out = []
    for col in wanted:
        if col not in frame.columns:
            continue
        v = frame[col].astype(str).str.strip().str.lower()
        if ((v != "") & (v != "nan")).any():
            out.append(col)
    return out

def num(row, col):
    try:
        return float(row[col])
    except Exception:
        return None

shown = filled(df, PHYS)
display(df[shown])
missing = [x for x in PHYS if x not in df.columns]
empty   = [x for x in PHYS if x in df.columns and x not in shown]
if missing: print("CSV 에 없는 컬럼:", ", ".join(missing))
if empty:   print("값이 전부 비어 표에서 뺀 컬럼:", ", ".join(empty))

for _, r in df.iterrows():
    bits = []
    cys = num(r, "cysteine_count")
    if cys is not None:
        bits.append(f"Cys {int(cys)}개 " + ("짝수 → disulfide 짝맞음"
                                            if int(cys) % 2 == 0 else
                                            "홀수 → unpaired cysteine (mispairing·응집 위험)"))
    met, trp = num(r, "methionine_count"), num(r, "tryptophan_count")
    if met is not None and trp is not None:
        bits.append(f"산화 후보 Met {int(met)} + Trp {int(trp)} = {int(met) + int(trp)}개")
    pi = num(r, "pI")
    if pi is not None:
        bits.append(f"pI {pi:.2f} ({'산성' if pi < 7 else '염기성'})")
    gr = num(r, "gravy")
    if gr is not None:
        bits.append(f"GRAVY {gr:+.3f} ({'친수성' if gr < 0 else '소수성'})")
    print(f"{r.get('id', '?')}:", " | ".join(bits))

odd = int(sum(1 for _, r in df.iterrows()
              if num(r, "cysteine_count") is not None and int(num(r, "cysteine_count")) % 2 == 1))
print(f"\n홀수 Cys 사슬 {odd}개 / 전체 {len(df)}개 — "
      + ("0개면 unpaired cysteine 위험 없음" if odd == 0
         else "1개 이상이면 mispairing·응집을 의심하고 어느 Cys 가 남는지 찾아야 해요"))
if "pI" in df.columns and len(df) > 1:
    spread = float(df["pI"].max()) - float(df["pI"].min())
    print(f"사슬 간 pI 차 {spread:.2f} — 1.0 이상이면 사슬 전하가 서로 반대쪽이라 "
          "charge symmetry(SFvCSP)를 구조 기반 도구로 따로 봐야 해요 (본문 8.3)")

## 3) liability motif — 몇 건이고 CDR 안인가 (본문 8.2)

본문 8.2 요약은 **"CDR 안에 산화·glyc·불안정 motif 가 몰리면 임상으로 가기 어렵다"** 고 해요.
그래서 hit 수만 세지 않고 **위치를 IMGT 번호로 바꿔 CDR/framework 에 귀속**합니다.
번호는 Ch.04 가 만든 numbering CSV 를 그대로 재활용해요(같은 서열일 때만 — 다르면 귀속을 생략합니다).

In [ ]:
import csv, pandas as pd

ALPHA = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
IMGT_CDR = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}   # IMGT CDR 정의

def read_fasta(path):
    d, n = {}, None
    for line in open(path):
        line = line.strip()
        if line.startswith(">"): n = line[1:].split()[0]; d[n] = ""
        elif n: d[n] += line
    return d

def imgt_positions(seq_id, seq):
    """Ch.04 numbering CSV → {서열 1-based 위치: IMGT 번호}. 서열이 다르면 (None, None)."""
    for fname in ("demo_imgt_H.csv", "demo_imgt_KL.csv"):
        for sub in ("my_run", "data"):
            p = ADV_ROOT / "04_numbering" / sub / fname
            if not p.exists():
                continue
            for row in csv.DictReader(open(p)):
                if row.get("Id") != seq_id:
                    continue
                start = int(row.get("seqstart_index") or 0)
                pos, built, i = {}, [], start
                for col in row:
                    key = str(col).strip()
                    if not key[:1].isdigit():        # 'Id'·'v_gene' 등 메타 컬럼은 건너뜀
                        continue
                    v = (row[col] or "").strip()
                    if v and v != "-":               # '-' 는 IMGT 갭
                        pos[i + 1] = int(key.rstrip(ALPHA))   # '111A' → 111 (insertion code)
                        built.append(v); i += 1
                if "".join(built) == seq[start:i]:
                    return pos, str(p)
    return None, None

def region_of(imgt_no):
    for name, (lo, hi) in IMGT_CDR.items():
        if lo <= imgt_no <= hi:
            return name
    return "framework"

seqs  = read_fasta("data/demo_mab.fa")
mcols = [m for m in MOTIF if m in df.columns]
hits  = []
for _, r in df.iterrows():
    for m in mcols:
        for q in str(r[m]).split(";"):
            if q.strip().isdigit():
                hits.append((r["id"], int(q), m))

print("검사한 motif:", ", ".join(m.replace("_", " ") for m in mcols) or "(컬럼 없음)")
print(f"총 hit {len(hits)}건")

if not hits:
    print("→ motif hit 0건이라 CDR/framework 로 귀속할 대상이 없어요. "
          "스캐너가 안 도는 게 아니라 이 후보에 hit 이 없는 거예요 (5절에서 검출되는 서열로 확인합니다).")
else:
    rows, no_number = [], []
    for sid in sorted({h[0] for h in hits}):
        pos_map, src = imgt_positions(sid, seqs.get(sid, ""))
        if pos_map is None:
            no_number.append(sid)
        else:
            print(f"[{sid} 번호 출처] {src}")
        for _sid, p, m in sorted(h for h in hits if h[0] == sid):
            imgt = pos_map.get(p) if pos_map else None
            rows.append({"id": sid, "pos": p, "residues": seqs.get(sid, "")[p - 1:p + 2],
                         "motif": m, "IMGT": imgt if imgt else "-",
                         "region": region_of(imgt) if imgt else "번호 없음"})
    hit_df = pd.DataFrame(rows)
    display(hit_df)
    if no_number:
        print("Ch.04 numbering 과 서열이 달라 IMGT 귀속을 못 한 사슬:", ", ".join(no_number),
              "→ Ch.04 노트북을 같은 FASTA 로 먼저 돌리면 채워집니다")
    in_cdr = int(hit_df["region"].isin(list(IMGT_CDR)).sum())
    fr     = int((hit_df["region"] == "framework").sum())
    print(f"CDR 안 {in_cdr}건 / framework {fr}건 / 번호 없음 {len(hit_df) - in_cdr - fr}건")
    print("판정 — " + ("CDR 안 hit 은 결합 부위 자체가 화학적으로 변할 수 있어 우선 처리 대상이에요."
                     if in_cdr else
                     "CDR 안 hit 0건 — framework hit 은 제조·보관 조건에서 다시 보되 결합력 위험은 낮아요."))

## 4) 내 결과 확인 — developability 개요 그래프 (본문 8.2)

2×2 패널 — 좌상 pI(주황, pH 7 기준선) · 우상 GRAVY(청록, 0 기준선) ·
좌하 Cys 개수(보라, 짝수면 짝맞음) · 우하 liability motif 누적 막대.

In [ ]:
from antibody_viz import liability_overview
from IPython.display import Image, display

png = "my_run/08_liability_overview.png"
liability_overview(resolve("liability.csv"),
                   "Developability — liability scan (demo mAb, 내 실행 결과)", png)
display(Image(png))

def panel(col, fmt):
    if col not in df.columns:
        return "(컬럼 없음)"
    out = []
    for _, r in df.iterrows():
        v = num(r, col)
        out.append(f"{r.get('id', '?')} " + (fmt.format(v) if v is not None else "-"))
    return " · ".join(out)

print("좌상 pI —",    panel("pI", "{:.2f}"))
print("우상 GRAVY —", panel("gravy", "{:+.3f}"))
print("좌하 Cys —",   panel("cysteine_count", "{:.0f}"))
if not hits:
    print(f"우하 motif 패널은 **막대 없이 범례만** 보여요 — 쌓을 값이 {len(hits)}건이기 때문이에요. "
          "빈 축 = 그리기 실패가 아니라 hit 0건이라는 뜻입니다.")
    print("판정 — 여러분 서열로 바꿔 돌려서 우하에 막대가 솟으면, 3절의 CDR/framework 귀속부터 확인하세요.")
else:
    print(f"우하 motif 패널 막대 높이 합 = {len(hits)}건 (사슬별로 누적).")
    print("판정 — 막대가 있는 사슬은 3절 표에서 그 hit 이 CDR 안인지부터 보세요.")

## 5) 입력 robustness — 모호 잔기(X/B/Z)가 섞여도 죽는가 (본문 8.1)

본문 8.1 은 QC 입력에 `X`·`B`·`Z` 가 섞이면 Biopython `ProteinAnalysis` 가 예외로 죽는 문제를
`liability_scan.py` 가 `ambiguous_residues` 컬럼으로 분리해 해결했다고 해요.
demo 서열에는 모호 잔기가 없어 그 컬럼이 비어 있었으니, **일부러 X·B·Z 와 홀수 Cys, motif 4종을 전부 넣은
짧은 합성 서열**로 확인합니다. 위 판정 코드가 "깨끗하지 않은" 입력에서도 맞는 말을 하는지 같이 봐요.

In [ ]:
import pathlib, pandas as pd

edge_fa = pathlib.Path("my_run/edge_case.fa")
edge_fa.write_text(">edge_case\nDIQMTQNGSCNSTDGKXBZWCVQLC\n")   # X·B·Z + Cys 3개(홀수) + motif 4종
run_tool([PY, SCRIPTS/"liability_scan.py", str(edge_fa), "--out", "my_run/liability_edge.csv"])

edge_csv = pathlib.Path("my_run/liability_edge.csv")
if not edge_csv.exists():
    print("합성 서열 스캔 산출물이 없어요 → 위 실행 로그를 확인하세요 (biopython 설치 여부).")
else:
    edge = pd.read_csv(edge_csv)
    display(edge[filled(edge, PHYS + MOTIF)])
    r = edge.iloc[0]
    ecys = int(r["cysteine_count"])
    ehits = sum(1 for m in MOTIF if m in edge.columns
                for q in str(r[m]).split(";") if q.strip().isdigit())
    print(f"모호 잔기 {r['ambiguous_residues']} 가 섞여 있어도 예외 없이 "
          f"pI {float(r['pI']):.2f} · GRAVY {float(r['gravy']):+.3f} 가 나왔어요 (표준 20종만으로 계산).")
    print(f"Cys {ecys}개 → " + ("짝수" if ecys % 2 == 0 else "홀수 = unpaired cysteine 경고")
          + f" | motif hit {ehits}건")
    print("판정 — 같은 코드가 demo 에서는 "
          f"{len(hits)}건, 이 합성 서열에서는 {ehits}건이라고 말해요. 판정문이 입력을 따라간다는 확인이에요.")

## 6) 이 스캔이 재는 축과 못 재는 축 (본문 8.1 · 8.3)

본문 8.1 은 developability 위험을 7종으로 정리했고, 본문 8.3 은 임상 분포와 비교하는 **TAP 5축**을 소개해요.
TAP 는 ABodyBuilder2 구조 + 임상 항체 분포가 필요해서 이 노트북(서열 전용)에서는 못 돌립니다.
무엇을 재고 무엇을 못 쟀는지 밝히는 것까지가 보고서예요.

In [ ]:
import pandas as pd
have = set(df.columns)

def cover(cols):
    return "O" if cols and all(x in have for x in cols) else "X"

risk = pd.DataFrame([
    ["Aggregation",       "hydrophobic patch·SAP", "gravy (사슬 평균 — patch 아님)",
     "부분" if "gravy" in have else "X"],
    ["Deamidation",       "motif regex",  "deamidation_NG, deamidation_NS",
     cover(["deamidation_NG", "deamidation_NS"])],
    ["Isomerization",     "motif regex",  "isomerization_DG",        cover(["isomerization_DG"])],
    ["Oxidation",         "Met/Trp count", "methionine_count, tryptophan_count",
     cover(["methionine_count", "tryptophan_count"])],
    ["N-glycosylation",   "N-X-S/T sequon", "N_glycosylation_NXS_T", cover(["N_glycosylation_NXS_T"])],
    ["Unpaired cysteine", "Cys 홀짝",     "cysteine_count, odd_cysteine_flag",
     cover(["cysteine_count", "odd_cysteine_flag"])],
    ["Charge/pI",         "pI 계산",      "pI",                      cover(["pI"])],
], columns=["본문 8.1 위험", "본문이 말한 스캔 방법", "이 CSV 의 컬럼", "이 노트북"])
display(risk)

tap = pd.DataFrame([
    ["CDR length",                    "부분 — 길이 자체는 Ch.04·Ch.09 에서 계산(임상 분포 대비는 못 함)"],
    ["Surface hydrophobicity (PSH)",  "X — 구조 표면 필요"],
    ["Patches of positive charge (PPC)", "X — 구조 표면 필요"],
    ["Patches of negative charge (PNC)", "X — 구조 표면 필요"],
    ["Heavy/light charge symmetry (SFvCSP)", "X — 구조 필요 (2절의 사슬 간 pI 차는 방향만 보는 신호)"],
], columns=["본문 8.3 TAP 5축", "이 노트북"])
display(tap)

full = int((risk["이 노트북"] == "O").sum())
print(f"판정 — 본문 8.1 의 7개 위험 중 {full}개는 서열만으로 확정, "
      f"{7 - full}개는 구조가 있어야 해요. TAP 5축은 0/5 — 이 노트북 결과만으로 "
      "\"developability 통과\"라고 쓰면 과장이에요.")
print("보완 경로 — TAP(웹) · aggregation predictor · hydrophobic/charge patch 시각화 · "
      "T-cell epitope 예측 (본문 8.3)")

## 7) 레퍼런스 대조 — 커밋된 스캔 결과와 같은가

In [ ]:
import pandas as pd, pathlib
mine_p = pathlib.Path("my_run/liability.csv")
if not mine_p.exists():
    print("my_run 스캔 결과가 없어 대조를 건너뜁니다 → 1절 실행 로그를 확인하세요.")
else:
    mine, ref = pd.read_csv(mine_p), pd.read_csv("data/liability.csv")
    common = [x for x in ref.columns if x in mine.columns]
    only   = [x for x in mine.columns if x not in ref.columns] + \
             [x for x in ref.columns if x not in mine.columns]
    same = len(mine) == len(ref) and mine[common].astype(str).equals(ref[common].astype(str))
    print(f"행 {len(mine)} vs {len(ref)} · 공통 컬럼 {len(common)}개 → "
          + ("모든 값이 같아요" if same else "값이 다른 컬럼이 있어요"))
    if only:
        print("한쪽에만 있는 컬럼:", ", ".join(only))
    if not same and len(mine) == len(ref):
        # 다른 컬럼만 골라 '내 결과 → 레퍼런스' 로 나란히 — 대괄호·따옴표 없이 읽히게.
        diff = []
        for col in common:
            if not mine[col].astype(str).equals(ref[col].astype(str)):
                for i in range(len(mine)):
                    m, rf = str(mine[col].iloc[i]), str(ref[col].iloc[i])
                    if m != rf:
                        diff.append({"컬럼": col, "행": i, "내 결과": m, "레퍼런스": rf})
        if diff:
            display(pd.DataFrame(diff))
    print("판정 — " + ("일치하면 같은 서열·같은 스캐너로 재현된 거예요."
                     if same else
                     "서열을 바꿔 돌렸다면 달라지는 게 정상이에요. 같은 demo 서열인데 다르면 "
                     "Biopython 버전 차이(molecular_weight·pI 반올림)를 먼저 의심하세요."))

> 다음 → 본문 [09. repertoire & naturalness](../09_repertoire/09_repertoire.md)